# HEART H200 — Independent Chemistry Evidence for PI Slides

**Purpose:** scientifically test whether frozen general-pretrained HEART embeddings encode meaningful transferable chemistry. Does **not** retrain HEART, touch training source, modify checkpoint or submit a Slurm job.**Existing 10K results reused** whenever possible; external benchmarks require one new frozen checkpoint inference on external molecules.

**Evidence hierarchy**
1. Existing masked-atom/geometry validation (pretraining success)
2. Original 10K CLS nearest-neighbor chemical structure test (representation quality; no GPU)
3. **Independent experimental property prediction** on ESOL, FreeSolv, and Lipophilicity, using frozen HEART CLS + **Ridge linear probe** vs strong chemical baselines on scaffold-held-out splits
4. Optional: original checkpoint molecular 3D denoising examples (3 molecules only)

**Important:** the Ridge probe is supervised training of a **small linear head** on benchmark labels; it never updates HEART. A benchmark signal may not be chemically novel if its SMILES were present in the 18.84M pretraining corpus. Exact raw-SMILES overlap is checked when the prior audit hashes are available; canonical overlap would need a separate audit.

## 0. Paths and environment — read-only source checks

Notebook + scripts can be uploaded together under `/home/ypan/heart_liver/model_train/HEART_Chemistry_Slide_Evidence/`. Results live in a **new** directory beside previous `/project/yli4/ypan/heart_evaluation/general_audit_10k_v2/` results.

Change only `ROOT`/`REPO`/`CHECKPOINT` if your HPC mapping differs. This cell never initiates a model run.

In [ ]:
from pathlib import Path
import sys, os, json, importlib.util, subprocess
import numpy as np, pandas as pd

def first_present(paths, need_file=False):
    return next((p for p in map(Path,paths) if (p.is_file() if need_file else p.is_dir())), None)

NOTEBOOK_DIR=first_present([
    Path.cwd(), Path.cwd()/"HEART_Chemistry_Slide_Evidence",
    Path('/home/ypan/heart_liver/model_train/HEART_Chemistry_Slide_Evidence'),
    Path('/mnt/home/ypan/heart_liver/model_train/HEART_Chemistry_Slide_Evidence')])
if NOTEBOOK_DIR is None or not (NOTEBOOK_DIR/'chemistry_tests.py').is_file():
    for base in [Path.cwd(),Path.cwd().parent,Path('/home/ypan/heart_liver/model_train')]:
        p=base/'HEART_Chemistry_Slide_Evidence'
        if (p/'chemistry_tests.py').is_file():NOTEBOOK_DIR=p;break
assert (NOTEBOOK_DIR/'chemistry_tests.py').is_file(), 'Extract COMPLETE ZIP, not only .ipynb'
if str(NOTEBOOK_DIR) not in sys.path:sys.path.insert(0,str(NOTEBOOK_DIR))
import chemistry_tests as chemistry

REPO=first_present([
    '/scratch/ypan/heart_liver/model_train/heart-molecular-pretraining-v2',
    '/home/ypan/heart_liver/model_train/heart-molecular-pretraining-v2',
    '/mnt/home/ypan/heart_liver/model_train/heart-molecular-pretraining-v2',
    '/project/yli4/ypan/heart-molecular-pretraining-v2'])
DATA_DIR=first_present([
    '/project/yli4/ypan/UniMolReference/ligand_data/ligands',
    '/mnt/project/yli4/ypan/UniMolReference/ligand_data/ligands',
    '/scratch/ypan/heart_liver/model_train/UniMolReference/ligand_data/ligands',
    '/home/ypan/heart_liver/model_train/UniMolReference/ligand_data/ligands'])
UNICORE=first_present([
    '/scratch/ypan/heart_liver/model_train/third_party/Uni-Core',
    '/project/yli4/ypan/third_party/Uni-Core',
    '/mnt/project/yli4/ypan/third_party/Uni-Core',
    '/home/ypan/heart_liver/model_train/third_party/Uni-Core'])
CKPT_NAME='runs/h200/full/heart-h200-b448-v1/checkpoint_best.pt'
CHECKPOINT=first_present([
    Path('/project/yli4/ypan')/CKPT_NAME,
    Path('/mnt/project/yli4/ypan')/CKPT_NAME,
    Path('/scratch/ypan/heart_liver/model_train/heart-molecular-pretraining-v2')/CKPT_NAME,
    Path('/home/ypan/heart_liver/model_train/heart-molecular-pretraining-v2')/CKPT_NAME,
    Path('/scratch/ypan/heart_liver/model_train/heart-molecular-pretraining')/CKPT_NAME],need_file=True)
ROOT=first_present([
    '/project/yli4/ypan/heart_evaluation/general_audit_10k_v2',
    '/mnt/project/yli4/ypan/heart_evaluation/general_audit_10k_v2',
    '/home/ypan/heart_liver/model_train/heart_evaluation/general_audit_10k_v2',
    '/scratch/ypan/heart_liver/model_train/heart_evaluation/general_audit_10k_v2'])
if ROOT is None:
    raise FileNotFoundError('Cannot find EXISTING general_audit_10k_v2 results. Set ROOT to its actual location.')
OLD_RUNS=ROOT/'checkpoint_evaluation'
MANIFEST_DIR=first_present([ROOT/'validation_10k_seed17',ROOT/'manifests' ])
if MANIFEST_DIR is None:
    candidates=list(ROOT.glob('**/sample_metadata.csv'))
    MANIFEST_DIR=candidates[0].parent if candidates else None
AUDIT_DIR=ROOT/'full_ligand_audit'/'raw'
if not AUDIT_DIR.is_dir():
    candidates=list(ROOT.glob('**/train_raw_unique_sorted.txt'))
    AUDIT_DIR=candidates[0].parent if candidates else None
OUTPUT=ROOT/'chemistry_slide_evidence'  # New files only; existing files untouched.
BENCHMARK_CSV=OUTPUT/'benchmark_csv'
EXTERNAL_LMDB=OUTPUT/'benchmark_generated_lmdb'
EXTERNAL_EMB=OUTPUT/'external_embeddings'
BENCHMARK_OUT=OUTPUT/'independent_property_benchmarks'
SIM_OUT=OUTPUT/'structural_similarity'
FIG_OUT=OUTPUT/'slides_from_existing_10k'

for name,obj in [('Notebook scripts',NOTEBOOK_DIR),('Frozen model code',REPO),
    ('Original ligands',DATA_DIR),('Uni-Core',UNICORE),('H200 checkpoint',CHECKPOINT),
    ('10K results',OLD_RUNS),('10K manifest',MANIFEST_DIR),('Raw exact-audit digests',AUDIT_DIR),
    ('NEW result folder',OUTPUT)]:print(f'{name:27}: {obj} | exists={bool(obj and Path(obj).exists())}')
assert REPO and (REPO/'model'/'unimol'/'tasks'/'unimol_contrastive.py').is_file()
assert OLD_RUNS.is_dir() and (OLD_RUNS/'full_10k_batch8'/'evaluation_summary.json').exists()
assert MANIFEST_DIR and (MANIFEST_DIR/'sample_metadata.csv').is_file()
print('Current kernel:',sys.executable)
print('Torch CUDA:',__import__('torch').cuda.is_available())

## 1. Slide 1: use existing 10K evidence (no GPU, no rerun)

One bar panel for **masked-atom accuracy vs majority-element baseline**; another for 3D coordinate denoising. Also save per-atom-class accuracy. Uses existing `evaluation_summary.json` and `atom_token_class_accuracy.csv`.

In [ ]:
RUN_EXISTING_FIGURES=True
if RUN_EXISTING_FIGURES:
    results=chemistry.existing_10k_slide_figures(OLD_RUNS,FIG_OUT)
    display(pd.DataFrame([{ 'split':k,'n':v['n_molecules'],
     'masked_accuracy':v['masked_atom_accuracy'],
     'majority_baseline':v.get('masked_atom_majority_class_baseline_accuracy'),
     'pred_coord_RMSE':v['coord_pred_rmse'],
     'noisy_coord_RMSE':v['coord_noisy_rmse']} for k,v in results.items()]))
    print('PNG + vector PDF:',FIG_OUT)

## 2. Original 10K structure-retrieval evidence (no GPU)

Search for nearest neighbors in frozen CLS space, then calculate **Morgan Tanimoto structural similarity**. Compare to **size-matched random controls** (within ±2 heavy atoms). This tests whether embeddings contain chemical topology beyond just molecule size. It is a computational reference test, **not** independent experimental downstream validation. 400 anchors / 4,000 gallery, not a memory-heavy 10,000 × 10,000 matrix.

In [ ]:
RUN_STRUCTURE_RETRIEVAL=True
if RUN_STRUCTURE_RETRIEVAL:
    t=chemistry.similarity_evidence(OLD_RUNS/'full_10k_batch8',MANIFEST_DIR,SIM_OUT,n_anchor=400,n_gallery=4000,seed=17)
    display(t[['near_tanimoto','control_tanimoto','near_heavy_atom_delta']].describe().round(3))
    print('Slide figures and example molecular drawings:',SIM_OUT)

## 3. Obtain external experimental benchmarks (small CSV files)

We use the original [DeepChem MoleculeNet datasets](https://deepchem.readthedocs.io/en/stable/api_reference/moleculenet.html):

- `delaney-processed.csv` — experimental aqueous solubility (ESOL)
- `SAMPL.csv` — experimental hydration free energy (FreeSolv)
- `Lipophilicity.csv` — experimental logD

If the HPC blocks external downloads, open the three URLs printed in this cell **in your browser**, save the CSV files and upload them to `BENCHMARK_CSV`. The notebook has **no** dependency on DeepChem being installed. Only these raw CSVs are required.

In [ ]:
TRY_DOWNLOAD_BENCHMARKS=False    # Enable only if your Jupyter has internet access
missing=chemistry.get_benchmark_csvs(BENCHMARK_CSV,TRY_DOWNLOAD_BENCHMARKS)
if missing:
    print('Upload these three files with exactly the displayed names:')
    for name,url,dest in missing:print(name,'\n URL:',url,'\n SAVE TO:',dest)
else:print('All 3 experimental benchmark CSVs present. Ready for preparation.')

## 4. Prepare independent 3D inputs (RDKit ETKDG; CPU only)

All source CSV labels are kept separately; model LMDB contains only `smi`, `atoms`, `coordinates`. Build **one reproducible conformer** per canonical structure, skip failures and report counts. Write new `valid.lmdb` plus copy of the exact original `dict.txt` into the **NEW** benchmark folder. No original data are changed. Do not repeat if outputs exist.

In [ ]:
RUN_BUILD_BENCHMARK_INPUTS=False   # Set True once the 3 CSVs above exist
if RUN_BUILD_BENCHMARK_INPUTS:
    assert not chemistry.get_benchmark_csvs(BENCHMARK_CSV,False), 'Missing dataset files: run Section 3 first'
    assert DATA_DIR and (DATA_DIR/'dict.txt').is_file()
    from shutil import copy2
    # No overwrite of existing benchmark molecular records.
    if not (EXTERNAL_LMDB/'external_manifest.csv').exists():
        chemistry.molecules_from_benchmarks(BENCHMARK_CSV,EXTERNAL_LMDB,max_atoms=256)
    if not (EXTERNAL_LMDB/'dict.txt').exists():
        copy2(DATA_DIR/'dict.txt', EXTERNAL_LMDB/'dict.txt')
    print('Ready:',EXTERNAL_LMDB)
else:print('Skipped benchmark ETKDG construction. This section is opt-in.')

## 5. Frozen HEART checkpoint inference (once, direct GPU Jupyter, NO Slurm)

**This is not pretraining or fine-tuning.** Load the existing `checkpoint_best.pt` with strict keys, run `model.eval()` inside `torch.inference_mode()`, capture **raw encoder CLS view0** with a read-only hook. Use the original `unimol_contrastive` task to process the generated external LMDB. Original model uses two independently augmented views and batch=8; no new network architecture.

Set `RUN_EXTERNAL_INFERENCE=True` only after Section 4 finishes and the Jupyter kernel shows CUDA True. Results are saved so rerunning this cell reuses completed embeddings.

In [ ]:
RUN_EXTERNAL_INFERENCE=False  # EXPLICIT OPT-IN: needs GPU; no sbatch, never updates HEART weights
if RUN_EXTERNAL_INFERENCE:
    assert CHECKPOINT and CHECKPOINT.is_file()
    assert UNICORE and UNICORE.is_dir()
    assert (EXTERNAL_LMDB/'valid.lmdb').is_file() and (EXTERNAL_LMDB/'dict.txt').is_file()
    assert __import__('torch').cuda.is_available(), 'Switch Notebook to your heart_gpu kernel with GPU'
    evidence=chemistry.generate_external_embeddings(REPO,UNICORE,CHECKPOINT,EXTERNAL_LMDB,EXTERNAL_EMB,batch_size=8)
    display(evidence)
else:print('Frozen external checkpoint inference not run; previous 10K results remain untouched.')

## 6. Independent benchmark prediction — statistical linear probes (CPU only)

Test if frozen HEART CLS conveys information about **experimentally measured chemical properties**. For **three scaffold-disjoint 80/10/10 splits**, evaluate five predictors on identical splits: label mean, heavy atoms only, RDKit descriptor Ridge, Morgan-ECFP4 Ridge, **frozen HEART CLS Ridge**. Select Ridge `alpha` using validation only; report held-out test RMSE, R², MAE with variability. Also compare **20% vs 100% labeled train set**.

The Ridge head learns from benchmark labels; HEART encoder remains frozen. This is transfer evidence only if test RMSE is informative relative to baselines, and source overlap limitations are properly stated.

In [ ]:
RUN_EXTERNAL_BENCHMARKS=False  # True AFTER frozen external inference completes
if RUN_EXTERNAL_BENCHMARKS:
    assert (EXTERNAL_EMB/'external_embedding_summary.json').is_file(), 'Section 5 not complete'
    df=chemistry.external_property_evidence(EXTERNAL_LMDB,EXTERNAL_EMB,BENCHMARK_OUT,
      audit_dir=AUDIT_DIR, seeds=(17,31,53), fractions=(0.2,1.0))
    display(df[df.train_fraction==1.0].groupby(['benchmark','model'])['rmse'].agg(['mean','std']).round(3))
    print('All slide-ready figures + raw CSV:',BENCHMARK_OUT)
else:print('Benchmark probes not run. Note: this cell is the ONLY supervised step; it never trains HEART.')

## 7. Optional 3D example from original validation ligand LMDB (3 molecules only)

Visualize actual original target/noisy/recovered coordinates, using the frozen checkpoint and original source LMDB. All positions are **task-normalized**, only masked atoms are emphasized, and this is an explanatory 3D depiction — **not** a molecular bond/energy geometry validation. No requirement to run this for the external benchmark slide.

In [ ]:
RUN_3D_EXAMPLES=False
if RUN_3D_EXAMPLES:
    assert CHECKPOINT and DATA_DIR and UNICORE and __import__('torch').cuda.is_available()
    molecule_keys=chemistry.geometry_example(REPO,UNICORE,CHECKPOINT,DATA_DIR,MANIFEST_DIR,
        OUTPUT/'geometry_examples',n=3)
    print('Original validation LMDB indices:',molecule_keys)
else:print('3D examples optional; kept off to avoid rerunning checkpoint unnecessarily.')

## 8. Slide-ready outputs / decision rule

**Recommended main Slide A: “Pretraining captures atom identity and local geometry”**: `slides_from_existing_10k/slide_1_structure_geometry.pdf`, plus `slide_1_atom_class_accuracy.pdf`.

**Recommended main Slide B: “Frozen HEART embeddings transfer to experimental chemistry tasks”**: three `*_benchmark_RMSE.pdf` charts plus one `*_HEART_pred_vs_experiment.pdf` **only if the experimental result is competitive and overlap caveats are noted**. Include mean±SD across three scaffold splits and compare to RDKit and Morgan baselines.

**Backup evidence**: `structural_similarity/CLS_structure_retrieval.pdf` and `Example_neighbor_molecules.png`; report size-matched control and nearest-neighbor chemical similarity.

**Interpretation rule**: (1) pretraining success ≠ independent chemical transfer; (2) outcome can be positive, mixed or negative — no claims before actual experiment; (3) 3D conformers are generated, not experimentally determined; (4) raw-SMILES pretrain-overlap checks do **not** rule out canonical-equivalent overlap. The paper/presentation may say “generalizable chemistry” only to the extent supported by measured benchmark results on carefully separated molecules.

In [ ]:
files=list(OUTPUT.rglob('*.png'))+list(OUTPUT.rglob('*.pdf'))+list(OUTPUT.rglob('*.csv'))
if not files:
    print('No new figures yet. Run Sections 1–2 now; benchmark results need Sections 3–6.')
else:
    display(pd.DataFrame([{'file':str(p.relative_to(OUTPUT)),'KB':round(p.stat().st_size/1024,1)} for p in sorted(files)]))
print('Everything saved under:',OUTPUT)